# 🦅 PREDATOR Analytics v56.1.4-ELITE — Full Stack Deploy on Colab

**Стек**: PostgreSQL 16 + Redis 7 + FastAPI + Vite Frontend + Nginx + Cloudflare Tunnel

**Порядок запуску**: Виконайте клітинки **зверху вниз**. Клітинка 6 тримає сесію.

> ⚠️ Colab Free: сесія ~12 год. Після відключення все треба запустити знову.
---

In [ ]:
%%bash
# ═══════════════════════════════════════════════════════
# КЛІТИНКА 1: Системні залежності + GPU перевірка
# ═══════════════════════════════════════════════════════
echo '🔍 Перевірка GPU...'
nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader 2>/dev/null || echo '⚠️  GPU недоступний'

echo ''
echo '📦 Встановлення системних пакетів...'
apt-get update -qq
apt-get install -y -qq \
  postgresql-16 postgresql-client-16 \
  redis-server \
  nginx \
  curl wget git unzip \
  build-essential \
  python3-pip python3-venv \
  nodejs npm 2>&1 | tail -5

echo ''
echo '📦 Оновлення Node.js до v20...'
curl -fsSL https://deb.nodesource.com/setup_20.x | bash - -qq
apt-get install -y -qq nodejs 2>&1 | tail -3
node -v && npm -v

echo ''
echo '📦 Cloudflare тунель...'
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
chmod +x /usr/local/bin/cloudflared

echo ''
echo '✅ Система готова!'
echo "RAM: $(free -h | awk '/^Mem:/{print $2}') | Диск: $(df -h / | awk 'NR==2{print $4}') вільно"

In [ ]:
%%bash
# ═══════════════════════════════════════════════════════
# КЛІТИНКА 2: PostgreSQL - налаштування + схема PREDATOR
# ═══════════════════════════════════════════════════════
echo '🗄️  Налаштування PostgreSQL...'

# Запуск PostgreSQL
service postgresql start
sleep 3

# Налаштування користувача та БД
su -c "psql -c \"CREATE USER predator WITH PASSWORD 'predator_secret_2026';\"" postgres 2>/dev/null || true
su -c "psql -c \"CREATE DATABASE predator_db OWNER predator;\"" postgres 2>/dev/null || true
su -c "psql -c \"GRANT ALL PRIVILEGES ON DATABASE predator_db TO predator;\"" postgres 2>/dev/null || true

# Схема PREDATOR
su -c "psql -d predator_db" postgres << 'SQL'
-- Rozширення
CREATE EXTENSION IF NOT EXISTS pgcrypto;
CREATE EXTENSION IF NOT EXISTS pg_trgm;

-- Компанії
CREATE TABLE IF NOT EXISTS companies (
    id BIGSERIAL PRIMARY KEY,
    edrpou VARCHAR(10) UNIQUE NOT NULL,
    name TEXT NOT NULL,
    risk_score DECIMAL(5,2) DEFAULT 0,
    risk_level VARCHAR(20) DEFAULT 'stable',
    status VARCHAR(20) DEFAULT 'active',
    sector VARCHAR(100),
    created_at TIMESTAMPTZ DEFAULT NOW(),
    updated_at TIMESTAMPTZ DEFAULT NOW()
);

-- Декларації
CREATE TABLE IF NOT EXISTS declarations (
    id BIGSERIAL PRIMARY KEY,
    declaration_number VARCHAR(50) UNIQUE NOT NULL,
    declaration_date DATE NOT NULL,
    company_edrpou VARCHAR(10),
    company_name TEXT,
    product_code VARCHAR(20),
    product_name TEXT,
    value_usd DECIMAL(15,2),
    weight_kg DECIMAL(15,3),
    created_at TIMESTAMPTZ DEFAULT NOW()
);

-- Ризики
CREATE TABLE IF NOT EXISTS risk_entities (
    id BIGSERIAL PRIMARY KEY,
    entity_type VARCHAR(20) NOT NULL,
    entity_id VARCHAR(100) NOT NULL,
    risk_score DECIMAL(5,2) DEFAULT 0,
    risk_level VARCHAR(20) DEFAULT 'stable',
    factors JSONB DEFAULT '{}',
    created_at TIMESTAMPTZ DEFAULT NOW(),
    updated_at TIMESTAMPTZ DEFAULT NOW(),
    UNIQUE(entity_type, entity_id)
);

-- Демо-дані
INSERT INTO companies (edrpou, name, risk_score, risk_level, sector) VALUES
  ('38210342', 'ТОВ ЕНЕРДЖИ-ГРУП', 92, 'critical', 'Енергетика'),
  ('41092384', 'ПРАТ ТЕХНО-ВЕСТ', 75, 'high', 'Технології'),
  ('29384712', 'ПП ЛОГІСТИК-ЦЕНТР ПЛЮС', 45, 'medium', 'Логістика'),
  ('31049582', 'ТОВ МЕТАЛ-ПРОМ', 22, 'stable', 'Металургія'),
  ('42938104', 'ДП СИСТЕМА', 68, 'elevated', 'Держсектор')
ON CONFLICT (edrpou) DO NOTHING;

INSERT INTO declarations (declaration_number, declaration_date, company_name, company_edrpou, product_code, product_name, value_usd, weight_kg) VALUES
  ('UA100060/2024/001234', '2024-03-15', 'ТОВ ЕНЕРДЖИ-ГРУП', '38210342', '8517', 'Смартфони', 450000, 2500),
  ('UA100060/2024/001235', '2024-03-16', 'ПРАТ ТЕХНО-ВЕСТ', '41092384', '8703', 'Автомобілі', 380000, 45000),
  ('UA100060/2024/001236', '2024-03-17', 'ПП ЛОГІСТИК-ЦЕНТР ПЛЮС', '29384712', '2710', 'Нафтопродукти', 920000, 180000)
ON CONFLICT (declaration_number) DO NOTHING;

\echo 'Схема PREDATOR створена!'
SQL

echo '✅ PostgreSQL готовий!'
su -c "psql -d predator_db -c 'SELECT COUNT(*) as компанії FROM companies;'" postgres

In [ ]:
%%bash
# ═══════════════════════════════════════════════════════
# КЛІТИНКА 3: Redis + FastAPI Backend
# ═══════════════════════════════════════════════════════
echo '🔴 Запуск Redis...'
service redis-server start
sleep 2
redis-cli ping

echo ''
echo '🐍 Встановлення Python залежностей...'
pip install -q \
  fastapi uvicorn[standard] \
  sqlalchemy asyncpg psycopg2-binary \
  redis hiredis orjson \
  python-jose[cryptography] passlib[bcrypt] \
  httpx pydantic-settings python-multipart \
  2>&1 | tail -3

echo ''
echo '📝 Створення FastAPI backend...'
mkdir -p /opt/predator-api

cat > /opt/predator-api/main.py << 'PYEOF'
"""
PREDATOR Analytics - Core API (Colab Edition)
FastAPI backend v56.1.4-COLAB
"""
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
import psycopg2
import redis
import json
import random
from datetime import datetime
from typing import Optional

app = FastAPI(
    title="PREDATOR Analytics API",
    description="OSINT Platform for Ukrainian Customs Analytics",
    version="56.1.4-COLAB"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

DB_CONFIG = {
    "host": "localhost",
    "database": "predator_db",
    "user": "predator",
    "password": "predator_secret_2026"
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

r = redis.Redis(host='localhost', port=6379, decode_responses=True)

@app.get("/")
def root():
    return {"status": "PREDATOR Analytics v56.1.4-COLAB", "mode": "colab", "timestamp": datetime.now().isoformat()}

@app.get("/api/v1/azr/status")
def azr_status():
    return {
        "status": "operational",
        "mode": "colab",
        "version": "56.1.4-COLAB",
        "services": {"postgres": True, "redis": True, "ai": True},
        "timestamp": datetime.now().isoformat()
    }

@app.post("/api/v1/auth/login")
def login(credentials: dict):
    # Simplified auth for Colab demo
    users = {
        "admin": {"password": "predator2026", "role": "admin"},
        "analyst": {"password": "analyst123", "role": "analyst"}
    }
    username = credentials.get("username", "")
    password = credentials.get("password", "")
    if username in users and users[username]["password"] == password:
        return {
            "access_token": f"colab-token-{username}-{random.randint(10000,99999)}",
            "token_type": "bearer",
            "user": {"username": username, "role": users[username]["role"], "tenant_id": "colab"}
        }
    raise HTTPException(status_code=401, detail="Невірні дані")

@app.get("/api/v1/dashboard/overview")
def dashboard_overview():
    try:
        conn = get_db()
        cur = conn.cursor()
        cur.execute("SELECT COUNT(*) FROM companies")
        companies_count = cur.fetchone()[0]
        cur.execute("SELECT COUNT(*) FROM declarations")
        decl_count = cur.fetchone()[0]
        cur.execute("SELECT COALESCE(SUM(value_usd), 0) FROM declarations")
        total_value = float(cur.fetchone()[0])
        conn.close()
        return {
            "total_companies": companies_count,
            "active_declarations": decl_count,
            "total_value_usd": total_value,
            "risk_alerts": 3,
            "mode": "colab"
        }
    except Exception as e:
        return {"error": str(e), "mode": "colab"}

@app.get("/api/v1/diligence/risk-entities")
def risk_entities(limit: int = 10, offset: int = 0):
    try:
        conn = get_db()
        cur = conn.cursor()
        cur.execute(
            "SELECT id, edrpou, name, risk_score, risk_level, status, sector, created_at FROM companies ORDER BY risk_score DESC LIMIT %s OFFSET %s",
            (limit, offset)
        )
        rows = cur.fetchall()
        conn.close()
        return [
            {"id": str(r[0]), "edrpou": r[1], "name": r[2], "risk_score": float(r[3] or 0),
             "risk_level": r[4], "status": r[5], "sector": r[6], "created_at": str(r[7])}
            for r in rows
        ]
    except Exception as e:
        return [{"error": str(e)}]

@app.get("/api/v1/market/declarations")
def declarations(page: int = 1, page_size: int = 15):
    try:
        conn = get_db()
        cur = conn.cursor()
        offset = (page - 1) * page_size
        cur.execute(
            "SELECT id, declaration_number, declaration_date, company_name, company_edrpou, product_code, product_name, value_usd, weight_kg FROM declarations ORDER BY declaration_date DESC LIMIT %s OFFSET %s",
            (page_size, offset)
        )
        rows = cur.fetchall()
        cur.execute("SELECT COUNT(*) FROM declarations")
        total = cur.fetchone()[0]
        conn.close()
        items = [
            {"id": str(r[0]), "declaration_number": r[1], "declaration_date": str(r[2]),
             "company_name": r[3], "company_edrpou": r[4], "product_code": r[5],
             "product_name": r[6], "value_usd": float(r[7] or 0), "weight_kg": float(r[8] or 0)}
            for r in rows
        ]
        return {"items": items, "total": total, "page": page, "page_size": page_size}
    except Exception as e:
        return {"items": [], "error": str(e)}

@app.get("/api/v1/market/overview")
def market_overview():
    return {
        "overview": {
            "stats": {
                "total_declarations": 4218932, "declarations_change": 12.5,
                "total_value_usd": 12450000000, "value_change": 8.2,
                "active_companies": 15420, "companies_change": 4.1,
                "total_products": 89430, "products_change": 15.7
            },
            "top_products": [
                {"product_code": "8517", "product_name": "Смартфони та обладнання зв'язку", "total_value_usd": 450000000, "growth_rate": 22.4},
                {"product_code": "8703", "product_name": "Легкові автомобілі", "total_value_usd": 380000000, "growth_rate": -5.2},
                {"product_code": "2710", "product_name": "Нафтопродукти", "total_value_usd": 920000000, "growth_rate": 12.8}
            ]
        }
    }

@app.get("/health")
def health():
    return {"status": "ok", "version": "56.1.4-COLAB", "timestamp": datetime.now().isoformat()}
PYEOF

echo '✅ FastAPI backend створено!'

In [ ]:
%%bash
# ═══════════════════════════════════════════════════════
# КЛІТИНКА 4: Клонування і збірка Frontend
# ═══════════════════════════════════════════════════════
echo '📦 Клонування PREDATOR Frontend...'

# Клонуємо репозиторій (або створюємо мінімальний build)
# Замініть URL на ваш реальний репозиторій:
REPO_URL="https://github.com/dima1203oleg/predator-analytics.git"

if git clone --depth=1 "$REPO_URL" /opt/predator-repo 2>&1; then
    echo '✅ Репозиторій клоновано!'
    cd /opt/predator-repo/apps/predator-analytics-ui
    
    # Встановлення залежностей
    echo '📦 npm install...'
    npm install --silent 2>&1 | tail -3
    
    # Build з production API URL (буде замінено після отримання тунель URL)
    echo '🔨 npm run build...'
    VITE_API_URL=/api/v1 \
    VITE_MODE=colab \
    VITE_ENABLE_MOCK_API=false \
    npm run build 2>&1 | tail -10
    
    UI_BUILD_DIR="/opt/predator-repo/apps/predator-analytics-ui/dist"
    echo "✅ Frontend зібрано: $UI_BUILD_DIR"
else
    echo '⚠️ Репозиторій недоступний. Створюємо заглушку...'
    mkdir -p /opt/predator-ui
    cat > /opt/predator-ui/index.html << 'HTML'
<!DOCTYPE html>
<html lang="uk">
<head><meta charset="UTF-8"><title>PREDATOR Analytics v56.1.4</title>
<style>
  body { background: #020408; color: #fff; font-family: system-ui; display: flex; align-items: center; justify-content: center; min-height: 100vh; margin: 0; }
  .card { text-align: center; padding: 60px; border: 1px solid rgba(255,255,255,.1); border-radius: 32px; }
  h1 { font-size: 3rem; color: #dc2626; margin: 0 0 16px; }
  p { color: #94a3b8; font-size: 1.1rem; }
  .badge { display: inline-block; background: rgba(220,38,38,.1); border: 1px solid rgba(220,38,38,.3); color: #dc2626; padding: 4px 16px; border-radius: 100px; font-size: .75rem; margin-bottom: 24px; }
</style></head>
<body><div class="card">
  <div class="badge">PREDATOR v56.1.4 | COLAB MODE</div>
  <h1>🦅 PREDATOR</h1>
  <p>Analytics Platform активна на Google Colab</p>
  <p style="margin-top:16px;font-size:.85rem;color:#475569">Фронтенд: репозиторій не знайдено. Запустіть з вашого Mac.</p>
</div></body></html>
HTML
    UI_BUILD_DIR="/opt/predator-ui"
    echo "✅ Заглушка UI створена: $UI_BUILD_DIR"
fi

echo "export UI_BUILD_DIR=$UI_BUILD_DIR" > /tmp/predator_env.sh

In [ ]:
%%bash
# ═══════════════════════════════════════════════════════
# КЛІТИНКА 5: Nginx + Запуск всіх сервісів
# ═══════════════════════════════════════════════════════
source /tmp/predator_env.sh 2>/dev/null || UI_BUILD_DIR="/opt/predator-ui"

echo '⚙️  Налаштування Nginx...'
cat > /etc/nginx/sites-available/predator << NGINX
server {
    listen 3030;
    root $UI_BUILD_DIR;
    index index.html;

    location /api/ {
        proxy_pass http://localhost:8000;
        proxy_set_header Host \$host;
        proxy_set_header X-Real-IP \$remote_addr;
        add_header 'Access-Control-Allow-Origin' '*';
    }

    location /health {
        proxy_pass http://localhost:8000/health;
    }

    location / {
        try_files \$uri \$uri/ /index.html;
    }
}
NGINX

ln -sf /etc/nginx/sites-available/predator /etc/nginx/sites-enabled/predator
rm -f /etc/nginx/sites-enabled/default
nginx -t && service nginx restart
echo '✅ Nginx налаштовано на порту 3030'

echo ''
echo '🚀 Запуск FastAPI backend на порту 8000...'
cd /opt/predator-api
nohup uvicorn main:app --host 0.0.0.0 --port 8000 --workers 2 > /var/log/predator-api.log 2>&1 &
echo $! > /tmp/predator_api.pid
sleep 5

# Перевірка
if curl -s http://localhost:8000/health | python3 -c "import sys,json; d=json.load(sys.stdin); print('✅ API:', d['status'])" 2>/dev/null; then
    echo '✅ FastAPI backend запущено!'
else
    echo '⚠️ API не відповідає. Логи:'
    tail -20 /var/log/predator-api.log
fi

echo ''
echo '📊 Стан сервісів:'
echo "  PostgreSQL: $(service postgresql status | grep -o 'active (running)\|inactive' | head -1)"
echo "  Redis:      $(service redis-server status | grep -o 'active (running)\|inactive' | head -1)"
echo "  Nginx:      $(service nginx status | grep -o 'active (running)\|inactive' | head -1)"
echo "  FastAPI:    $(curl -s http://localhost:8000/health | python3 -c 'import sys,json; print(json.load(sys.stdin)[\"status\"])' 2>/dev/null || echo 'error')"

In [ ]:
import subprocess, time, re, threading

# ═══════════════════════════════════════════════════════
# КЛІТИНКА 6: Cloudflare тунель → публічний URL
# ═══════════════════════════════════════════════════════

print('🌐 Відкриття Cloudflare тунелю на порт 3030 (Frontend + API)...')

cf_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:3030'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

public_url = None
for _ in range(60):
    try:
        line = cf_proc.stdout.readline().decode('utf-8', errors='replace')
        match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group(0)
            break
    except:
        break

if public_url:
    print('\n' + '='*65)
    print('🦅  PREDATOR Analytics — COLAB DEPLOYMENT АКТИВНИЙ!')
    print('='*65)
    print(f'\n🌐  Frontend: {public_url}')
    print(f'⚙️   API:      {public_url}/api/v1')
    print(f'🏥  Health:   {public_url}/health')
    print('\n' + '─'*65)
    print('👇 ВСТАВТЕ У .env.local на вашому Mac:')
    print('─'*65)
    print(f'VITE_API_URL={public_url}/api/v1')
    print(f'VITE_BACKEND_PROXY_TARGET={public_url}')
    print(f'VITE_ENABLE_MOCK_API=false')
    print(f'VITE_MODE=remote')
    print('─'*65)
    print('\n📌 Логін: admin / predator2026')
    print('='*65)
else:
    print('❌ URL не отримано. Перевірте чи cloudflared встановлено (Клітинка 1).')
    print('Логи:')
    out, _ = cf_proc.communicate(timeout=5)
    print(out.decode('utf-8', errors='replace')[:500])

In [ ]:
# ═══════════════════════════════════════════════════════
# КЛІТИНКА 7: Keep-Alive (тримає сесію активною)
# НЕ ЗУПИНЯЙТЕ ЦЮ КЛІТИНКУ — вона тримає Colab alive
# ═══════════════════════════════════════════════════════
import time, datetime, httpx

print(f'🦅 PREDATOR Colab Gateway активний')
print(f'📡 URL: {public_url}')
print(f'⏱️  Сесія почалась: {datetime.datetime.now().strftime("%H:%M:%S")}')
print('Ця клітинка підтримує сесію. Натисніть ⏹️ для зупинки.\n')

start = datetime.datetime.now()
check_count = 0

while True:
    elapsed = datetime.datetime.now() - start
    h = int(elapsed.total_seconds() // 3600)
    m = int((elapsed.total_seconds() % 3600) // 60)
    
    check_count += 1
    if check_count % 5 == 0:  # Кожні 5 хвилин перевіряємо API
        try:
            r = httpx.get('http://localhost:8000/health', timeout=3)
            api_status = '✅'
        except:
            api_status = '⚠️ '
    else:
        api_status = '✅'
    
    print(f'\r⏱️ {h}г {m:02d}хв | API {api_status} | {datetime.datetime.now().strftime("%H:%M:%S")}', end='', flush=True)
    time.sleep(60)